[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C61_Detection_Practice_Interview_Course/03_debug/03_debug_playbook.ipynb)

# 03 · 训练与部署调试手册（单 batch 过拟合 / NaN 溯源 / 三类管线 bug 的指纹）

目标：把「模型不 work」这句没信息量的话，拆成一串**可以在 10 分钟内证伪的假设**。

本 notebook 你会亲手实现：
1. **一个真的能训练的迷你检测头**（纯 numpy，手写反传 + 数值梯度对拍）
2. **「先把一个 batch 过拟合」诊断法**，以及它的四种失败形态各自意味着什么
3. **静态 IoU 分配下小目标正样本数 = 0** 的数值证据（TSR 的头号隐形杀手）
4. **NaN 的四条通路**：朴素 BCE 的溢出断点、退化框的 0/0、坏标注的 $\log(w\le0)$，
   以及一个 `trace_nonfinite` 逐段定位器
5. **梯度/激活健康检查器**：非有限值、梯度爆炸/消失、死亡神经元
6. **三类经典数据管线 bug 的构造与指纹检测**
   —— 类别 ID 偏移 / 坐标格式弄反 / 通道顺序 BGR-RGB，每种给出**独有指纹**与一行验证
7. **诊断树（playbook）的代码化**：症状 + 观察到的事实 → 排好序的假设与检查动作

> 心智模型：**调试的进展不是「试了很多东西」，而是「排除了很多可能」。**

## 1 · 一个真的能训练的迷你检测头

结构就是检测头的最小骨架：`输入特征 → ReLU 隐层 → 两个分支`
- **分类分支**：一个 logit，判「这个位置是不是前景」（BCE）
- **回归分支**：4 维框偏移 $(dx, dy, dw, dh)$，**只对正样本算**（Smooth L1）

手写反传，然后用**数值梯度对拍**——这就是第 1 节说的「反传段的不变量」。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)


def sigmoid(z):
    # 数值稳定版：正负分支分开算，避免 exp 溢出
    return np.where(z >= 0, 1.0 / (1.0 + np.exp(-np.abs(z))),
                    np.exp(-np.abs(z)) / (1.0 + np.exp(-np.abs(z))))


def stable_bce(z, y):
    # log-sum-exp 稳定形式：max(z,0) - z*y + log(1+exp(-|z|))
    return np.maximum(z, 0.0) - z * y + np.log1p(np.exp(-np.abs(z)))


def init_model(d_in=12, d_h=48, seed=0):
    r = np.random.default_rng(seed)
    return dict(W1=r.normal(0, np.sqrt(2 / d_in), (d_in, d_h)), b1=np.zeros(d_h),
                w2=r.normal(0, np.sqrt(2 / d_h), (d_h,)), b2=np.zeros(1),
                W3=r.normal(0, np.sqrt(2 / d_h), (d_h, 4)), b3=np.zeros(4))


def forward(p, X):
    z1 = X @ p['W1'] + p['b1']
    h = np.maximum(z1, 0.0)                       # ReLU
    return dict(z1=z1, h=h,
                logit=h @ p['w2'] + p['b2'][0],   # 分类分支
                box=h @ p['W3'] + p['b3'])        # 回归分支


def loss_grads(p, X, y, t, w_box=1.0):
    n = len(X)
    f = forward(p, X)
    l_cls = float(stable_bce(f['logit'], y).mean())
    pos = y > 0.5
    npos = int(pos.sum())
    d = f['box'][pos] - t[pos] if npos else np.zeros((0, 4))
    ad = np.abs(d)
    l_box = float(np.where(ad < 1.0, 0.5 * d ** 2, ad - 0.5).mean()) if npos else 0.0
    loss = l_cls + w_box * l_box

    dlogit = (sigmoid(f['logit']) - y) / n                       # dL/dlogit
    dbox = np.zeros_like(f['box'])
    if npos:
        dbox[pos] = w_box * np.where(ad < 1.0, d, np.sign(d)) / (npos * 4)
    dh = np.outer(dlogit, p['w2']) + dbox @ p['W3'].T
    dz1 = dh * (f['z1'] > 0)                                     # ReLU 的导数
    g = dict(W1=X.T @ dz1, b1=dz1.sum(0), w2=f['h'].T @ dlogit,
             b2=np.array([dlogit.sum()]), W3=f['h'].T @ dbox, b3=dbox.sum(0))
    return loss, g, f, dict(l_cls=l_cls, l_box=l_box, n_pos=npos)


# —— 不变量：解析梯度 == 数值梯度 ——
Xc = rng.normal(0, 1, (16, 12))
yc = (rng.random(16) > 0.5).astype(float)
tc = rng.normal(0, 1, (16, 4))
pc = init_model(seed=1)
_, Gc, _, _ = loss_grads(pc, Xc, yc, tc)
worst = 0.0
for k in pc:
    flat = pc[k].ravel()
    for idx in rng.choice(flat.size, min(4, flat.size), replace=False):
        e, old = 1e-6, flat[idx]
        flat[idx] = old + e; Lp = loss_grads(pc, Xc, yc, tc)[0]
        flat[idx] = old - e; Lm = loss_grads(pc, Xc, yc, tc)[0]
        flat[idx] = old
        num, ana = (Lp - Lm) / (2 * e), Gc[k].ravel()[idx]
        worst = max(worst, abs(num - ana) / max(abs(num), abs(ana), 1e-8))
        assert np.isclose(num, ana, rtol=3e-4, atol=1e-8), (k, idx, num, ana)
print(f'✅ 数值梯度对拍通过，最大相对误差 {worst:.2e}')
print('   这就是「反传段的不变量」—— 它应该作为单元测试常驻仓库，而不是出事才写。')

## 2 · 万能诊断法：先把一个 batch 过拟合

24 个样本，关掉增强/正则/衰减，反复训练。**一个有足够容量的模型必须把训练 loss 压到接近 0。**

这句话是**无条件成立**的——它不依赖数据质量、不依赖任务难度。
所以**做不到就一定有 bug**，而不是「这个任务比较难」。

In [ ]:
def train(p, X, y, t, steps=2000, lr=0.05, frozen=(), wd=0.0, record_every=0):
    # 极简 Adam。frozen 里的参数不更新；wd = weight decay（做过拟合诊断时必须设 0）
    p = {k: v.copy() for k, v in p.items()}
    m = {k: np.zeros_like(v) for k, v in p.items()}
    v = {k: np.zeros_like(x) for k, x in p.items()}
    hist = []
    for s in range(1, steps + 1):
        loss, g, _, info = loss_grads(p, X, y, t)
        if record_every and (s == 1 or s % record_every == 0):
            hist.append((s, loss, info['l_cls'], info['l_box']))
        for k in p:
            if k in frozen:
                continue
            gg = g[k] + wd * p[k]
            m[k] = 0.9 * m[k] + 0.1 * gg
            v[k] = 0.999 * v[k] + 0.001 * gg ** 2
            p[k] -= lr * (m[k] / (1 - 0.9 ** s)) / (np.sqrt(v[k] / (1 - 0.999 ** s)) + 1e-8)
    loss, _, _, info = loss_grads(p, X, y, t)
    return p, loss, info, hist


Xb = rng.normal(0, 1, (24, 12))
yb = (rng.random(24) > 0.5).astype(float)
tb = rng.normal(0, 1, (24, 4))
p0 = init_model(seed=2)
L_init = loss_grads(p0, Xb, yb, tb)[0]

_, l_ok, i_ok, hist = train(p0, Xb, yb, tb, steps=2000, lr=0.05, record_every=400)
print(f'初始 loss = {L_init:.4f}   （正样本 {i_ok["n_pos"]} / {len(Xb)}）')
print(f'{"step":>6s}{"loss":>12s}{"cls":>12s}{"box":>12s}')
for s, l, lc, lbx in hist:
    print(f'{s:>6d}{l:>12.3e}{lc:>12.3e}{lbx:>12.3e}')
assert l_ok < 1e-3, l_ok
print(f'\n✅ 单 batch 过拟合成功（final loss {l_ok:.2e}）')
print('   => 数据 / 编码 / 前向 / 损失 / 反传 / 优化器 **整块可用**。')
print('   接下来所有问题只可能出在：数据规模、增强、学习率调度、正则、验证管线。')
print('   **一个实验砍掉半个搜索空间 —— 没有别的实验有这么高的信息量/成本比。**')

In [ ]:
# 四种「过拟合不了」的形态，各自意味着什么
def verdict(loss, l_init, tol=1e-3):
    if loss < tol:
        return '✅ 正常：前六段全部可用'
    if abs(loss - l_init) < 1e-9:
        return '❌ loss 一动不动 -> 梯度没流到权重（requires_grad / optimizer / no_grad / lr=0）'
    if loss > 0.6 * l_init:
        return '⚠️  几乎不动 -> lr 过小 或 warmup 还没结束 或 大部分参数被冻'
    return '⚠️  停在非零平台 -> 模型只能输出常数（weight decay 没关 / 死亡神经元 / 容量不足）'


cases = []
_, l_all, _, _ = train(p0, Xb, yb, tb, steps=300, lr=0.05, frozen=tuple(p0.keys()))
cases.append(('全部参数都冻住了', l_all))
_, l_bias, _, _ = train(p0, Xb, yb, tb, steps=2000, lr=0.05,
                        frozen=('W1', 'b1', 'w2', 'W3', 'b3'))
cases.append(('只有输出 bias 可训练', l_bias))
_, l_lr, _, _ = train(p0, Xb, yb, tb, steps=2000, lr=2e-6)
cases.append(('学习率小了 4 个数量级', l_lr))
_, l_wd, _, _ = train(p0, Xb, yb, tb, steps=2000, lr=0.05, wd=0.5)
cases.append(('忘了关 weight decay', l_wd))
cases.append(('正常', l_ok))

print(f'{"场景":<24s}{"final loss":>12s}   判定')
for nm, l in cases:
    print(f'{nm:<24s}{l:>12.4f}   {verdict(l, L_init)}')

assert abs(l_all - L_init) < 1e-12, '全冻结时 loss 必须逐位不变'
assert l_bias > 0.6 * L_init and l_lr > 0.6 * L_init
assert 1e-3 < l_wd < 0.8 * L_init
print('\n⚠️  做过拟合诊断前必须关掉：增强 / dropout / weight decay / **lr warmup 与衰减** / EMA。')
print('   最常见的自欺欺人是 warmup 没走完就下结论 —— 那时 lr 还接近 0，看起来像「梯度不流」。')
print('⚠️  另一个坑：检测的 loss 是多项之和，某一项塌成 0 而另一项没动，总和看起来也会很小。')
print('   **必须分项打印，并且必须同时看 mAP。**')

## 3 · 检测特有的陷阱：正样本数 = 0

在有标签分配的检测器里，「过拟合一个 batch」不是平凡的。
**如果分配策略把所有 anchor 都判成负样本，分类分支只会学到「全是背景」，loss 会降到某个平台然后不动。**

下面用 RetinaNet 的默认 anchor 配置（P3–P7，stride 8/16/32/64/128，
base size 32/64/128/256/512，每级 3 个尺度）算一算：
**多大的交通标志才能拿到至少一个 IoU ≥ 0.5 的正样本 anchor？**

In [ ]:
def iou_mat(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    bb = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / np.maximum(aa[:, None] + bb[None, :] - inter, 1e-9)


def make_anchors(levels, img=256):
    out = []
    for stride, base in levels:
        for s in [1.0, 2 ** (1 / 3), 2 ** (2 / 3)]:          # 每级 3 个尺度
            sz = base * s
            cs = np.arange(stride / 2, img, stride)
            cx, cy = np.meshgrid(cs, cs)
            cx, cy = cx.ravel(), cy.ravel()
            out.append(np.stack([cx - sz / 2, cy - sz / 2, cx + sz / 2, cy + sz / 2], 1))
    return np.concatenate(out, 0)


P3P7 = [(8, 32), (16, 64), (32, 128), (64, 256), (128, 512)]
A = make_anchors(P3P7)
A2 = make_anchors([(4, 16)] + P3P7)                            # 额外加一个 P2 层
print(f'anchor 总数：P3-P7 = {len(A)}，加 P2 后 = {len(A2)}')
print(f'\n{"标志边长":>9s}{"P3-P7 最大IoU":>14s}{"正样本数":>9s}{"加P2 最大IoU":>14s}{"正样本数":>9s}')
zero_pos = []
for gs in [8, 12, 16, 20, 24, 32, 48, 64, 96]:
    g = np.array([[128 - gs / 2, 128 - gs / 2, 128 + gs / 2, 128 + gs / 2]])
    v, v2 = iou_mat(A, g)[:, 0], iou_mat(A2, g)[:, 0]
    n1, n2 = int((v >= 0.5).sum()), int((v2 >= 0.5).sum())
    if n1 == 0:
        zero_pos.append(gs)
    print(f'{gs:>9d}{v.max():>14.3f}{n1:>9d}{v2.max():>14.3f}{n2:>9d}')

assert zero_pos == [8, 12, 16, 20], zero_pos
assert int((iou_mat(A2, np.array([[122., 122., 134., 134.]]))[:, 0] >= 0.5).sum()) > 0
print(f'\n🚨 **≤ {max(zero_pos)} px 的交通标志在 P3-P7 上拿不到任何一个正样本 anchor。**')
print('   它对分类损失完全不可见 —— 模型不是「学不好」，是**根本没被要求去学**。')
print('   TSR 里 80 米外的限速牌在 1920 宽的图上约 15-20 px，缩到 640 输入后只剩 5-7 px。')
print('   => 所以「单 batch 过拟合」在检测里**必须同时打印每步分到的正样本数**；')
print('      正样本数 = 0 时，问题在分配策略，不在网络。')
print('   => 修法：加 P2 层（上表右侧）/ 降低小目标的 IoU 阈值 / 换 center-based 分配')
print('      / 换尺度不敏感的度量（C57 模块 03 的 NWD）。')

## 4 · NaN 的四条通路

按「第几个 iteration 出现」分诊：
- **第 0–1 步就出现** ⇒ 数值实现或数据问题（本节的 ②③）
- **训练几百步后出现** ⇒ 发散（①）
- **固定 seed 后总在同一步出现** ⇒ 某个特定样本（③）

In [ ]:
def naive_bce(z, y):
    p = 1.0 / (1.0 + np.exp(-z))                 # 朴素实现：先算概率再取 log
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


print('② log(0)：朴素 BCE 在 y=0 时的溢出断点')
print(f'{"z":>5s}{"float32 朴素":>16s}{"float64 朴素":>16s}{"稳定版":>12s}')
brk32 = brk64 = None
with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
    for z in [5., 10., 16., 17., 20., 36., 37., 40., 90.]:
        n32 = float(naive_bce(np.float32(z), np.float32(0.0)))
        n64 = float(naive_bce(np.float64(z), 0.0))
        st = float(stable_bce(np.float64(z), 0.0))
        if brk32 is None and not np.isfinite(n32):
            brk32 = z
        if brk64 is None and not np.isfinite(n64):
            brk64 = z
        f32 = 'inf' if not np.isfinite(n32) else f'{n32:.4f}'
        f64 = 'inf' if not np.isfinite(n64) else f'{n64:.4f}'
        print(f'{z:>5.0f}{f32:>16s}{f64:>16s}{st:>12.4f}')
print(f'\n断点：float32 在 z≈{brk32:.0f}，float64 在 z≈{brk64:.0f}；稳定版永远有限。')
assert brk32 == 17 and brk64 == 37
assert np.isfinite(stable_bce(np.float64(1e4), 0.0))
# 稳定版与朴素版在安全区内必须完全一致
zs = np.linspace(-10, 10, 41)
assert np.allclose(stable_bce(zs, 0.0), naive_bce(zs, 0.0), atol=1e-12)
assert np.allclose(stable_bce(zs, 1.0), naive_bce(zs, 1.0), atol=1e-12)
print('✅ 两者在安全区数学等价；差别只在极端 z 上，而极端 z 恰恰是训练发散时必然出现的。')

In [ ]:
# ② 除零：退化框（标注里 x1 == x2）
def iou_pair_1(a, b, eps=0.0):
    a, b = np.asarray(a, float), np.asarray(b, float)
    inter = (np.clip(np.minimum(a[2], b[2]) - np.maximum(a[0], b[0]), 0, None) *
             np.clip(np.minimum(a[3], b[3]) - np.maximum(a[1], b[1]), 0, None))
    u = ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)
    return inter / (u + eps)


degenerate = np.array([10., 10., 10., 20.])          # 宽度为 0：标注员点一下就保存了
with np.errstate(divide='ignore', invalid='ignore'):
    v_bad = iou_pair_1(degenerate, degenerate)
    v_ok = iou_pair_1(degenerate, degenerate, eps=1e-9)
    # ③ 坏标注：框回归目标 log(w / w_anchor)
    t_zero = np.log(np.float64(0.0) / 32)
    t_neg = np.log(np.float64(-5.0) / 32)
print(f'退化框的 IoU：无 eps -> {v_bad}   有 eps -> {v_ok}')
print(f'log(w/w_a)：w=0 -> {t_zero}   w=-5 -> {t_neg}')
assert np.isnan(v_bad) and v_ok == 0.0
assert np.isinf(t_zero) and np.isnan(t_neg)


def trace_nonfinite(stages):
    # stages: [(名字, 张量), ...]，按前向顺序。返回第一个含 NaN/Inf 的段
    for name, arr in stages:
        a = np.asarray(arr, float)
        if not np.isfinite(a).all():
            fin = a[np.isfinite(a)]
            return name, dict(n_nan=int(np.isnan(a).sum()), n_inf=int(np.isinf(a).sum()),
                              lo=float(fin.min()) if fin.size else float('nan'),
                              hi=float(fin.max()) if fin.size else float('nan'))
    return None, {}


with np.errstate(divide='ignore', invalid='ignore'):
    pipeline = [('input', np.ones((4, 3))),
                ('backbone', np.ones((4, 8)) * 2.0),
                ('cls_logit', np.array([1.2, -0.4, 3.1, 0.0])),
                ('box_target', np.log(np.array([32., 16., 0., 64.]) / 32)),   # ← 第 3 个框宽为 0
                ('loss', np.array([1.0]))]
    where, det = trace_nonfinite(pipeline)
print(f'\n第一个出问题的段：{where}  {det}')
assert where == 'box_target' and det['n_inf'] == 1
print('✅ 关键在于**抓第一现场**：NaN 会传染，两三步之后整个网络都是 NaN，那时信息量为零。')
print('   工程做法：每 N 步常驻记录 grad norm、各层 max|x|、**loss 的各分项分开记**。')
print('   只有 box 分项炸而 cls 正常 => 几乎必然是坏标注；所有分项同时炸 => 才是学习率。')

## 5 · 梯度与激活的健康检查器

四个检查项，全部可以在训练循环里低成本常驻：
**非有限值 / 梯度爆炸 / 梯度消失 / 死亡神经元**（整个 batch 上激活恒为 0 的隐层单元）。

In [ ]:
def health_check(acts, grads, gn_hi=1e3, gn_lo=1e-7, dead_hi=0.5):
    issues = []
    for n, a in acts.items():
        if not np.isfinite(np.asarray(a, float)).all():
            issues.append(f'nonfinite_act:{n}')
    for n, g in grads.items():
        if not np.isfinite(np.asarray(g, float)).all():
            issues.append(f'nonfinite_grad:{n}')
    fin = [np.asarray(g, float) for g in grads.values() if np.isfinite(np.asarray(g, float)).all()]
    gn = float(np.sqrt(sum(float((g ** 2).sum()) for g in fin))) if fin else float('nan')
    if np.isfinite(gn):
        if gn > gn_hi:
            issues.append('grad_explosion')
        elif gn < gn_lo:
            issues.append('grad_vanish')
    for n, a in acts.items():
        a = np.asarray(a, float)
        if a.ndim == 2 and np.isfinite(a).all():
            dead = float((np.abs(a) <= 1e-12).all(axis=0).mean())   # 整个 batch 上恒为 0
            if dead > dead_hi:
                issues.append(f'dead_units:{n}')
    return sorted(issues), gn


_, G_h, F_h, _ = loss_grads(p0, Xb, yb, tb)
scen = {}
scen['健康'] = ({'h': F_h['h']}, G_h)
scen['梯度爆炸'] = ({'h': F_h['h']}, {k: v * 1e7 for k, v in G_h.items()})
p_dead = {k: v.copy() for k, v in p0.items()}
p_dead['b1'] = np.full_like(p_dead['b1'], -50.0)      # 巨大的负 bias -> ReLU 全灭
_, G_d, F_d, _ = loss_grads(p_dead, Xb, yb, tb)
scen['死亡神经元'] = ({'h': F_d['h']}, G_d)
G_n = {k: v.copy() for k, v in G_h.items()}; G_n['W1'][0, 0] = np.nan
scen['梯度里有 NaN'] = ({'h': F_h['h']}, G_n)

print(f'{"场景":<16s}{"grad norm":>13s}   诊断')
for nm, (a, g) in scen.items():
    iss, gn = health_check(a, g)
    print(f'{nm:<16s}{gn:>13.3e}   {iss if iss else "无异常"}')

assert health_check(*scen['健康'])[0] == []
assert 'grad_explosion' in health_check(*scen['梯度爆炸'])[0]
assert 'dead_units:h' in health_check(*scen['死亡神经元'])[0]
assert 'nonfinite_grad:W1' in health_check(*scen['梯度里有 NaN'])[0]
print('\n👉 「死亡神经元」在检测里有一个很有辨识度的后果：**回归分支退化到输出常数**，')
print('   表现为所有预测框尺寸差不多、位置都靠近图像中心（= 训练集框分布的中位数）。')
print('   看到「所有框长得一样」，先怀疑模型压根没在学，而不是先怀疑标签分配。')

## 6 · 管线 bug 之一：类别 ID 偏移

**指纹：混淆矩阵的质量集中在偏离对角线 $k$ 格的次对角线上。**

来源千奇百怪但都很常见：COCO 的 `category_id` 从 1 开始、
有的框架把 background 放在 index 0 而有的不放、
两侧从不同来源构造类别列表、某个类在某个 split 里样本数为 0 导致后面全部左移。

**它不会让 loss 异常**——模型照样能学，只是学到了一个偏移一位的映射。

In [ ]:
NC = 8
CLASSES = ['限速30', '限速60', '限速80', '停车让行', '禁止左转', '注意行人', '解除限速', '指路牌']
r6 = np.random.default_rng(3)

true_cls = r6.integers(0, NC, 600)
pred_ok = true_cls.copy()
flip = r6.random(600) < 0.12                       # 12% 的正常分类错误
pred_ok[flip] = r6.integers(0, NC, int(flip.sum()))
pred_shift = (pred_ok + 1) % NC                    # ← 注入 bug：类别整体 +1


def conf_mat(t, p, n=NC):
    cm = np.zeros((n, n), int)
    for a, b in zip(t, p):
        cm[a, b] += 1
    return cm


CM_OK, CM_BAD = conf_mat(true_cls, pred_ok), conf_mat(true_cls, pred_shift)


def show_cm(cm, title):
    print(title)
    print(' ' * 10 + ''.join(f'{c[:4]:>7s}' for c in CLASSES))
    for i in range(NC):
        print(f'{CLASSES[i]:<10s}' + ''.join(f'{cm[i, j]:>7d}' for j in range(NC)))


show_cm(CM_BAD, '有 bug 时的混淆矩阵（行=真值，列=预测）：')
print('\n👉 质量整体跑到了对角线右边一格 —— 这就是「次对角线」指纹。')
print('   对比：正常模型的质量在对角线上，错误弥散分布。')
print(f'   正常：对角线占比 {np.trace(CM_OK) / CM_OK.sum():.1%}   '
      f'有 bug：对角线占比 {np.trace(CM_BAD) / CM_BAD.sum():.1%}')
assert np.trace(CM_BAD) / CM_BAD.sum() < 0.05 and np.trace(CM_OK) / CM_OK.sum() > 0.8
print('\n⚠️  注意 mAP 会精确掉到 ~0 而不是掉一半：AP 是逐类算的，')
print('   偏移之后类 c 的预测全落进类 c+1 的评测里，两者在图像上完全不重叠 => 每类 AP 都是 0。')
print('   **「mAP 恰好是 0 而不是 0.3」本身就是线索 —— 模型能力问题不会让指标精确归零。**')

## 7 · 管线 bug 之二：坐标格式弄反

四个数就是四个数，任何格式都能塞进去，**不会报错**。

**指纹**（下表会实测）：
- `cxcywh` / `xywh` 被当成 `xyxy`：宽或高常常是负的 → clip 后面积为 0 →
  **IoU 恰好等于 0 的比例接近 100%**（模型定位不准时这个比例几乎是 0）
- `yxyx`（TF 系）被当成 `xyxy`：**`corr(pred_cx, gt_cy)` 很高而 `corr(pred_cx, gt_cx)` 很低**
  —— 中心点散点图被转置了

In [ ]:
r7 = np.random.default_rng(5)
n = 400
cx = r7.uniform(60, 1860, n); cy = r7.uniform(60, 1020, n)
sz = np.exp(r7.normal(np.log(34), 0.5, n))                 # TSR 的典型尺寸分布
GT7 = np.stack([cx - sz / 2, cy - sz / 2, cx + sz / 2, cy + sz / 2], 1)

PREDS = {
    '正常（框略有偏差）': GT7 + r7.normal(0, 0.02 * sz[:, None], (n, 4)),
    '定位很差（真·模型问题）': GT7 + np.repeat(r7.normal(0, 0.30 * sz[:, None], (n, 2)), 2, axis=1),
    'cxcywh 当 xyxy': np.stack([cx, cy, sz, sz], 1),
    'xywh 当 xyxy': np.stack([cx - sz / 2, cy - sz / 2, sz, sz], 1),
    'yxyx 当 xyxy': GT7[:, [1, 0, 3, 2]],
}

CANDS = {
    'as_is': lambda b: b,
    'xywh->xyxy': lambda b: np.stack([b[:, 0], b[:, 1], b[:, 0] + b[:, 2], b[:, 1] + b[:, 3]], 1),
    'cxcywh->xyxy': lambda b: np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                                        b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1),
    'yxyx->xyxy': lambda b: b[:, [1, 0, 3, 2]],
}


def iou_pair(a, b):
    x1 = np.maximum(a[:, 0], b[:, 0]); y1 = np.maximum(a[:, 1], b[:, 1])
    x2 = np.minimum(a[:, 2], b[:, 2]); y2 = np.minimum(a[:, 3], b[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    bb = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / np.maximum(aa + bb - inter, 1e-9)


def box_fingerprint(pred, gt):
    v = iou_pair(pred, gt)
    ap = np.clip(pred[:, 2] - pred[:, 0], 0, None) * np.clip(pred[:, 3] - pred[:, 1], 0, None)
    ag = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    pcx, pcy = (pred[:, 0] + pred[:, 2]) / 2, (pred[:, 1] + pred[:, 3]) / 2
    gcx, gcy = (gt[:, 0] + gt[:, 2]) / 2, (gt[:, 1] + gt[:, 3]) / 2
    return dict(iou=float(v.mean()), zero=float((v < 1e-9).mean()),
                area=float(np.median(ap) / np.median(ag)),
                cxx=float(np.corrcoef(pcx, gcx)[0, 1]), cxy=float(np.corrcoef(pcx, gcy)[0, 1]))


print(f'{"情形":<24s}{"平均IoU":>9s}{"IoU=0占比":>11s}{"面积比":>9s}{"corr(px,gx)":>13s}{"corr(px,gy)":>13s}')
FP = {}
for nm, pb in PREDS.items():
    FP[nm] = box_fingerprint(pb, GT7)
    f = FP[nm]
    print(f'{nm:<24s}{f["iou"]:>9.3f}{f["zero"]:>11.0%}{f["area"]:>9.2f}{f["cxx"]:>13.2f}{f["cxy"]:>13.2f}')

assert FP['定位很差（真·模型问题）']['zero'] < 0.05, '模型定位不准时几乎没有精确为 0 的 IoU'
assert FP['cxcywh 当 xyxy']['zero'] > 0.95 and FP['xywh 当 xyxy']['zero'] > 0.95
assert FP['yxyx 当 xyxy']['cxy'] > 0.9 > FP['yxyx 当 xyxy']['cxx']
assert FP['正常（框略有偏差）']['cxx'] > 0.9 > FP['正常（框略有偏差）']['cxy']
print('\n👉 「定位很差」和「格式弄反」在 mAP 上都会很难看，但**指纹完全不同**：')
print(f'   定位差：平均 IoU {FP["定位很差（真·模型问题）"]["iou"]:.2f}，'
      f'**IoU 恰好为 0 的比例 {FP["定位很差（真·模型问题）"]["zero"]:.0%}**（框再歪也总有些许重叠）')
print(f'   格式错：平均 IoU {FP["cxcywh 当 xyxy"]["iou"]:.2f}，'
      f'**IoU 恰好为 0 的比例 {FP["cxcywh 当 xyxy"]["zero"]:.0%}**（宽或高为负，被 clip 成 0 面积）')
print('   => 区分它们的不是均值，是**「恰好为 0」的那个比例**，以及面积比是否塌成 0。')
print('👉 x/y 互换的指纹是**相关系数被转置**：corr(pred_cx, gt_cy) 反而接近 1。')
print('   这个 bug 在方形图上很难发现，在 1920x1080 上会让框飞出画面。')

## 8 · 管线 bug 之三：通道顺序 BGR / RGB

这一类和前两类有本质区别：**它不会让指标归零，只会掉 5–15 个点**。
于是你会以为「这次训练没调好」，然后在一个有 bug 的管线上做三个月的架构消融。

**TSR 特有的指纹**：交通标志的颜色<u>就是</u>语义（红=禁令、蓝=指示、黄=警告），
R/B 互换会把红牌变成蓝牌；而白底黑字的指路牌 $R\approx G\approx B$，几乎不受影响。
=> **掉点具有类别选择性**，这正是它和「归一化 mean/std 写错」（所有类均匀掉点）的区别。

In [ ]:
SIGN_GROUPS = [
    ('红色禁令(限速/禁止)', np.array([0.75, 0.25, 0.22]), 'color'),
    ('蓝色指示(直行/环岛)', np.array([0.20, 0.35, 0.78]), 'color'),
    ('黄色警告(注意行人)', np.array([0.85, 0.75, 0.20]), 'color'),
    ('白底黑字(指路牌)', np.array([0.82, 0.82, 0.81]), 'neutral'),
    ('灰色解除限速', np.array([0.46, 0.47, 0.48]), 'neutral'),
]
PRIOR = np.array([0.40, 0.18, 0.12, 0.20, 0.10])          # 红色禁令占比最高（限速+禁止）


def make_patches(n, seed):
    # 合成 16x16 的标志块：底色 + 光照增益 + 噪声 + 中间较暗的图案区
    r = np.random.default_rng(seed)
    lab = r.choice(len(SIGN_GROUPS), n, p=PRIOR)
    P = np.zeros((n, 16, 16, 3))
    for i, c in enumerate(lab):
        img = np.tile(SIGN_GROUPS[c][1], (16, 16, 1)) * r.uniform(0.88, 1.12)
        img += r.normal(0, 0.035, (16, 16, 3))
        img[5:11, 5:11] *= r.uniform(0.25, 0.5)
        P[i] = np.clip(img, 0, 1)
    return P, lab


TR, ytr = make_patches(1500, 1)
TE, yte = make_patches(600, 2)
mean_rgb = lambda P: P.reshape(len(P), -1, 3).mean(1)
CENTROID = np.stack([mean_rgb(TR)[ytr == k].mean(0) for k in range(len(SIGN_GROUPS))])
predict = lambda P: ((mean_rgb(P)[:, None, :] - CENTROID[None]) ** 2).sum(-1).argmin(1)

COLOR_IDS = [k for k, g in enumerate(SIGN_GROUPS) if g[2] == 'color']
NEUTRAL_IDS = [k for k, g in enumerate(SIGN_GROUPS) if g[2] == 'neutral']


def report(P, y, tag):
    pr = predict(P)
    print(f'[{tag}]  总体准确率 {(pr == y).mean():.3f}')
    for k, (nm, _, grp) in enumerate(SIGN_GROUPS):
        m = y == k
        top = SIGN_GROUPS[int(np.bincount(pr[m], minlength=len(SIGN_GROUPS)).argmax())][0]
        print(f'   {nm:<20s}[{grp:7s}] acc={(pr[m] == y[m]).mean():.3f}  最常被判成 -> {top}')
    mc, mn = np.isin(y, COLOR_IDS), np.isin(y, NEUTRAL_IDS)
    ac, an = float((pr[mc] == y[mc]).mean()), float((pr[mn] == y[mn]).mean())
    print(f'   >>> 颜色语义类 acc={ac:.3f}    中性色类 acc={an:.3f}')
    return float((pr == y).mean()), ac, an


a_rgb = report(TE, yte, 'RGB 正确')
print()
a_bgr = report(TE[..., ::-1], yte, 'BGR 通道顺序弄反')
assert a_rgb[1] > 0.95 and a_rgb[2] > 0.95
assert a_bgr[1] < 0.05, '颜色语义类应该崩到接近 0'
assert a_bgr[2] > 0.95, '中性色类几乎不受影响 —— 这就是指纹'
print('\n🔎 **指纹：颜色语义类 1.00 -> 0.00，中性色类 1.00 -> 1.00。**')
print('   对照：如果是归一化 mean/std 写错，所有类别会**均匀**掉点，')
print('   因为它不改变通道之间的相对关系，只是整体平移/缩放。')

In [ ]:
def detect_channel_swap(batch, ref_mean_rgb, ratio=0.5):
    # 把 batch 的通道均值与训练集统计比：反过来更接近就说明通道被交换了
    o = np.asarray(batch).reshape(-1, 3).mean(0)
    d0 = float(np.linalg.norm(o - ref_mean_rgb))
    d1 = float(np.linalg.norm(o[::-1] - ref_mean_rgb))
    return (d1 < d0 * ratio), d0, d1


REF = mean_rgb(TR).mean(0)
print('训练集通道均值 ref =', np.round(REF, 3))
for tag, batch in [('RGB 输入', TE), ('BGR 输入', TE[..., ::-1])]:
    sw, d0, d1 = detect_channel_swap(batch, REF)
    print(f'{tag:<10s} ||obs-ref||={d0:.4f}  ||reversed(obs)-ref||={d1:.4f}  -> 疑似交换: {sw}')
assert detect_channel_swap(TE, REF)[0] is False
assert detect_channel_swap(TE[..., ::-1], REF)[0] is True

# 最强的确诊手段：把输入反过来重测，指标恢复即坐实
a_fix = report(TE[..., ::-1][..., ::-1], yte, '把输入通道反过来 -> 恢复')
assert abs(a_fix[0] - a_rgb[0]) < 1e-12
print('\n✅ 一行验证：`x = x[..., ::-1]` 之后指标完全恢复 => 确诊通道顺序。')
print('⚠️  组合 bug 会污染指纹：训练用 PIL(RGB)+cv2.resize，推理用 cv2.imread(BGR)+PIL.resize，')
print('   两个 bug 叠加时你看到的是「颜色类的小目标崩得特别厉害」，很容易误判成小目标问题。')
print('   **所以指纹检查必须逐项独立：一次只改一个变量重测。**（模块 01 的单变量原则）')

## 9 · 诊断树（playbook）的代码化

每条假设带四个字段：**名字 / 检查动作 / 若成立会观察到什么 / 先验权重**，
再加一个 `when(facts)` 把「你已经观察到的事实」变成权重调整。

**输出是排好序的检查清单**，而不是一个答案——调试本来就是逐步排除。

In [ ]:
PLAYBOOK = {
    'loss_nan': [
        dict(name='① 学习率过大 / 数值溢出', prior=3.0,
             check='看 NaN 之前 200 步的 grad norm 与各层 max|x| 曲线',
             expect='NaN 之前有一段指数上升',
             when=lambda f: 3.0 if f.get('first_nan_iter', 0) > 50 else 0.2),
        dict(name='② log(0) / 除零（自写损失缺 eps、退化框）', prior=3.0,
             check='对同一 batch 只跑 cls 分项、再只跑 box 分项；检查标注里有无 w<=0 或 h<=0',
             expect='某一分项单独就能触发；换成 log-sum-exp 稳定版后消失',
             when=lambda f: 4.0 if f.get('first_nan_iter', 999) <= 2 else 0.5),
        dict(name='③ 坏标注（负宽高 / 越界坐标 / 空框）', prior=2.0,
             check='固定 seed 定位到具体 iteration，dump 那个 batch 的文件名',
             expect='每次都在同一步 NaN，且只有 box 分项炸',
             when=lambda f: 4.0 if f.get('same_iter_every_run') else 0.6),
        dict(name='④ 反传中的除零（sqrt(0) / 零向量归一化 / GIoU 闭包为 0）', prior=1.5,
             check='前向逐层挂哨兵，再单独检查 .grad',
             expect='前向所有中间张量有限，只有梯度非有限',
             when=lambda f: 4.0 if (f.get('forward_finite') and f.get('grad_nonfinite')) else 1.0),
    ],
    'loss_flat': [
        dict(name='单 batch 都过拟合不了 -> 前六段有 bug', prior=5.0,
             check='24 个样本、关掉增强/正则/衰减，训 2000 步',
             expect='loss 必须 < 1e-3；否则按本 notebook 第 2 节的四种形态分诊',
             when=lambda f: 5.0 if not f.get('can_overfit_one_batch', True) else 0.1),
        dict(name='学习率不合适', prior=3.0, check='lr range test（1e-7 -> 10 指数扫描）',
             expect='取「下降最陡」处的 1/10',
             when=lambda f: 1.0),
        dict(name='标签映射错', prior=2.5, check='把编码后的目标解码回框与类别名，画到图上',
             expect='解码结果与原 GT 逐元素一致', when=lambda f: 1.0),
        dict(name='数据没打乱', prior=2.0, check='打印前 200 个样本的类别序列；查 shuffle/sampler',
             expect='loss 曲线的周期 = 一个 epoch 或一个分片',
             when=lambda f: 3.0 if f.get('loss_periodic') else 0.5),
        dict(name='冻错了层', prior=2.0,
             check='统计可训练参数量；对比 optimizer.param_groups；打印 BN 的 running_mean 是否在变',
             expect='可训练参数量远小于预期', when=lambda f: 1.0),
    ],
    'map_zero': [
        dict(name='类别 ID 偏移 0/1', prior=5.0, check='把预测类别整体 ±1 重算 mAP',
             expect='mAP 突然正常；混淆矩阵的质量在偏移 k 格的次对角线上', when=lambda f: 1.0),
        dict(name='坐标格式弄反', prior=4.5,
             check='把预测框按 cxcywh/xywh/yxyx -> xyxy 各试一遍，重算平均 IoU',
             expect='某个变换下平均 IoU 从 0.0x 跳到 0.8+；且原始 IoU=0 的比例接近 100%',
             when=lambda f: 1.0),
        dict(name='评测集与训练集类别表不一致', prior=4.0,
             check='diff 两侧类别列表的**顺序**，不只是集合；查是否有类在某个 split 里样本数为 0',
             expect='混淆矩阵是一个置换矩阵', when=lambda f: 1.0),
        dict(name='模型能力不足', prior=0.5, check='看训练 loss 与训练集上的 mAP',
             expect='训练集 mAP 也很低', when=lambda f: 1.0),
    ],
    'val_bad': [
        dict(name='验证管线与训练不一致（伪过拟合）', prior=4.0,
             check='**把训练集喂进验证的那条代码路径**跑评测',
             expect='训练集在验证路径上同样崩 => 与泛化无关',
             when=lambda f: 5.0 if f.get('train_as_val_also_bad') else 0.2),
        dict(name='忘了 model.eval()（BN 用了 batch 统计）', prior=2.0,
             check='对比 eval/train 模式下同一批数据的输出', expect='两者差异巨大', when=lambda f: 1.0),
        dict(name='真·过拟合', prior=2.0, check='看验证 loss 是否先降后升；加数据/加增强做消融',
             expect='有明确拐点，加数据能改善',
             when=lambda f: 3.0 if f.get('val_loss_has_turning_point') else 0.8),
        dict(name='捷径特征（跨采集批次失效）', prior=1.5,
             check='按采集批次给验证集分桶，比较桶内与跨桶指标',
             expect='同批次好、跨批次崩', when=lambda f: 1.0),
    ],
}


def diagnose(symptom, facts=None):
    facts = facts or {}
    items = [(h['prior'] * h['when'](facts), h) for h in PLAYBOOK[symptom]]
    items.sort(key=lambda x: -x[0])
    return [dict(score=round(s, 2), name=h['name'], check=h['check'], expect=h['expect'])
            for s, h in items]


def show(symptom, facts, title):
    print(f'\n=== {title} ===')
    for i, h in enumerate(diagnose(symptom, facts), 1):
        print(f'{i}. [{h["score"]:>5.2f}] {h["name"]}\n     查：{h["check"]}\n     若成立：{h["expect"]}')


show('loss_nan', {'first_nan_iter': 1}, 'NaN 出现在第 1 个 iteration')
show('loss_nan', {'first_nan_iter': 800}, 'NaN 出现在第 800 个 iteration')
show('map_zero', {}, 'mAP 恒为 0')
show('val_bad', {'train_as_val_also_bad': True}, '训练好、验证崩，且训练集走验证路径也崩')

assert diagnose('loss_nan', {'first_nan_iter': 1})[0]['name'].startswith('② log(0)')
assert diagnose('loss_nan', {'first_nan_iter': 800})[0]['name'].startswith('① 学习率')
assert diagnose('map_zero')[0]['name'].startswith('类别 ID 偏移')
assert diagnose('map_zero')[-1]['name'] == '模型能力不足', '「模型不行」永远排最后'
assert '验证管线' in diagnose('val_bad', {'train_as_val_also_bad': True})[0]['name']
assert '过拟合' in diagnose('val_bad', {'val_loss_has_turning_point': True})[0]['name']
print('\n✅ 注意 map_zero 里「模型能力不足」的先验只有 0.5 —— **它永远排在最后**。')

## ✏️ 练习 1：数值稳定的 BCE

实现 `my_bce(z, y)`，要求：
- 在 $|z| \le 10$ 的安全区里与朴素实现 $-y\log\sigma(z)-(1-y)\log(1-\sigma(z))$ 数值一致（`atol=1e-12`）
- 在 $|z|$ 极大（如 $10^4$）时**仍然有限**
- 支持数组输入

提示：$\mathcal{L} = \max(z,0) - zy + \log(1+e^{-|z|})$。

In [ ]:
def my_bce(z, y):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
zz = np.linspace(-10, 10, 81)
for yy in [0.0, 1.0]:
    assert np.allclose(my_bce(zz, yy), naive_bce(zz, yy), atol=1e-12), yy
for big in [1e2, 1e4, -1e4]:
    v0, v1 = float(my_bce(np.float64(big), 0.0)), float(my_bce(np.float64(big), 1.0))
    assert np.isfinite(v0) and np.isfinite(v1), big
# y=0 时 loss 应等于 max(z,0)+log(1+e^-|z|)，z 很大时约等于 z 本身
assert abs(float(my_bce(np.float64(1e4), 0.0)) - 1e4) < 1e-6
assert float(my_bce(np.float64(1e4), 1.0)) < 1e-6
# 与 float32 朴素版对比：朴素版在 z=17 就废了
with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
    assert not np.isfinite(float(naive_bce(np.float32(17.0), np.float32(0.0))))
assert np.isfinite(float(my_bce(np.float32(17.0), np.float32(0.0))))
print('✅ 练习 1 通过：稳定版在安全区数学等价，在极端 z 上不溢出。')
print('   这不是「精度更好」的问题 —— 朴素版给出的是 inf/nan，是**完全错误**。')

## ✏️ 练习 2：类别 ID 偏移检测器

实现 `detect_label_shift(cm)`：输入 $C\times C$ 混淆矩阵（行=真值，列=预测），
返回 `(k, mass)`：
- `k` 使 $\sum_i cm[i,\,i{+}k]$ 最大的偏移量（**不做环绕**，只累加下标合法的项）
- `mass` = 该次对角线上的样本数 ÷ 总样本数

`k == 0` 说明正常；`k != 0` 且 `mass` 很高就是中招了。

In [ ]:
def detect_label_shift(cm):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
k0, m0 = detect_label_shift(CM_OK)
k1, m1 = detect_label_shift(CM_BAD)
print(f'正常模型   -> k={k0}, 次对角线质量={m0:.1%}')
print(f'类别 +1 bug -> k={k1}, 次对角线质量={m1:.1%}')
assert k0 == 0 and m0 > 0.8
assert k1 == 1 and m1 > 0.6
# 手算：一个 3x3 的纯 -1 偏移矩阵
tiny = np.array([[0, 0, 0], [7, 0, 0], [0, 9, 0]])
assert detect_label_shift(tiny) == (-1, 1.0), detect_label_shift(tiny)
# 全零矩阵不能崩
assert detect_label_shift(np.zeros((4, 4), int))[1] == 0.0
print('✅ 练习 2 通过：一个混淆矩阵 + 一行代码，就能把「mAP=0」的头号元凶钉死。')

## ✏️ 练习 3：坐标格式自动识别

实现 `best_box_format(pred, gt, cands=CANDS)`：对每个候选变换算变换后与 `gt` 的**平均 IoU**，
返回 `(最佳变换名, {变换名: 平均IoU})`。可直接用上面的 `iou_pair`。

In [ ]:
def best_box_format(pred, gt, cands=None):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
expect = {'正常（框略有偏差）': 'as_is', '定位很差（真·模型问题）': 'as_is',
          'cxcywh 当 xyxy': 'cxcywh->xyxy', 'xywh 当 xyxy': 'xywh->xyxy',
          'yxyx 当 xyxy': 'yxyx->xyxy'}
print(f'{"情形":<24s}{"识别出的格式":<16s}{"该格式下的平均IoU":>18s}')
for nm, pb in PREDS.items():
    best, sc = best_box_format(pb, GT7)
    print(f'{nm:<24s}{best:<16s}{sc[best]:>18.3f}')
    assert best == expect[nm], (nm, best)
    if nm.endswith('当 xyxy'):
        assert sc[best] > 0.99, '修正后应当几乎完美重合'
        assert sc['as_is'] < 0.05, '未修正时平均 IoU 接近 0'
print('✅ 练习 3 通过：坐标格式不该靠「读代码猜」，应该靠「试一遍看 IoU」。')
print('   根治手段：在数据结构里带上格式标签（Boxes(fmt=...)），转换必须显式调用。')

## ✏️ 练习 4：症状 + 事实 → 排序后的检查清单

实现 `next_checks(symptom, facts, top=2)`：调用 `diagnose`，返回**前 `top` 条**的
`(name, check)` 二元组列表。再用它验证四个场景的首选检查是否符合预期。

In [ ]:
def next_checks(symptom, facts=None, top=2):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
c1 = next_checks('loss_nan', {'first_nan_iter': 1}, top=2)
assert len(c1) == 2 and isinstance(c1[0], tuple) and len(c1[0]) == 2
assert c1[0][0].startswith('② log(0)')
assert next_checks('loss_nan', {'first_nan_iter': 800})[0][0].startswith('① 学习率')
assert next_checks('map_zero', {})[0][0].startswith('类别 ID 偏移')
assert '验证管线' in next_checks('val_bad', {'train_as_val_also_bad': True})[0][0]
assert next_checks('loss_flat', {'can_overfit_one_batch': False})[0][0].startswith('单 batch')

print('把本 notebook 的观察串成一次完整分诊：')
FACTS = dict(first_nan_iter=1, can_overfit_one_batch=False,
             train_as_val_also_bad=True, loss_periodic=False)
for sym, title in [('loss_nan', 'loss 变 NaN'), ('loss_flat', 'loss 不降'),
                   ('map_zero', 'mAP 恒为 0'), ('val_bad', '验证极差')]:
    print(f'\n[{title}] 先做这两件事：')
    for nm, ck in next_checks(sym, FACTS, top=2):
        print(f'   · {nm}\n     -> {ck}')
print('\n✅ 练习 4 通过：调试的产物是**排好序的检查清单**，不是一个猜测。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_bce(z, y):
    z = np.asarray(z, dtype=np.float64) if np.ndim(z) else np.float64(z)
    return np.maximum(z, 0.0) - z * y + np.log1p(np.exp(-np.abs(z)))

In [ ]:
# 练习 2 参考答案
def detect_label_shift(cm):
    cm = np.asarray(cm)
    n = cm.shape[0]
    tot = int(cm.sum())
    best_k, best_m = 0, -1
    for k in range(-n + 1, n):
        m = int(sum(cm[i, i + k] for i in range(n) if 0 <= i + k < n))
        if m > best_m:
            best_k, best_m = k, m
    return best_k, (best_m / tot if tot else 0.0)

In [ ]:
# 练习 3 参考答案
def best_box_format(pred, gt, cands=None):
    cands = CANDS if cands is None else cands
    sc = {name: float(iou_pair(fn(pred), gt).mean()) for name, fn in cands.items()}
    return max(sc, key=lambda k: sc[k]), sc

In [ ]:
# 练习 4 参考答案
def next_checks(symptom, facts=None, top=2):
    return [(h['name'], h['check']) for h in diagnose(symptom, facts)[:top]]

---
## 🧪 真实工程胶囊：调试手册（可原样贴进团队 wiki）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 检测模型调试手册 —— 症状 -> 检查顺序（每条都给「若成立会观察到什么」）
# ══════════════════════════════════════════════════════════════════════

# ── 0. 常驻的不变量测试（写成 pytest，不是出事才手写）──────────────────
#   test_encode_decode_roundtrip : decode(encode(box)) == box            (atol=1e-6)
#   test_loss_on_perfect_pred    : L(GT_as_pred, GT) < 1e-6
#   test_gt_as_prediction        : evaluate(GT_as_pred, GT).mAP == 1.0
#   test_grad_numeric            : 解析梯度 vs 数值梯度  rtol < 1e-4
#   test_dataset_sanity          : 所有框 w>1 and h>1 and 0<=x1<x2<=W；断言里打印文件名
#   test_class_table_md5         : 训练/评测两侧 classes.json 的 md5 必须一致

# ── 1. 【最先做】单 batch 过拟合 ────────────────────────────────────────
#   8-32 个样本，**关掉**：增强 / dropout / weight decay / lr warmup 与衰减 / EMA
#   通过条件：训练 loss < 1e-3 且训练集 mAP > 0.95
#   检测特有：**每步打印分到的正样本数**；n_pos == 0 说明问题在标签分配，不在网络
#   四种失败形态：
#     loss 一动不动        -> requires_grad / optimizer.param_groups / no_grad / lr=0
#     loss 停在初值附近     -> lr 过小；warmup 还没走完；大部分参数被冻
#     loss 停在非零平台     -> weight decay 没关 / 死亡神经元 / 只有 bias 可训练
#     loss -> 0 但 mAP = 0 -> **训练侧对、评测侧错**，直接跳到第 3 节

# ── 2. loss = NaN ──────────────────────────────────────────────────────
#   第一个问题永远是「它是第几个 iteration 出现的」
#     0-1 步     -> 数值实现或数据（log(0)、退化框 0/0、log(w<=0)）
#     几百步后   -> 学习率/发散（看 grad norm 是否指数上升）
#     固定 seed 后总在同一步 -> 某个特定样本，dump 那个 batch 的文件名
#   手写损失一律用 log-sum-exp 稳定式：max(z,0) - z*y + log1p(exp(-|z|))
#     朴素式的断点：float32 z≈17，float64 z≈37
#   IoU / GIoU 的分母一律 + 1e-9（真实数据里 x1==x2 的退化框并不罕见）
#   常驻记录：grad norm、各层 max|x|、**loss 各分项分开记**
#     只有 box 分项炸 -> 坏标注；所有分项同时炸 -> 学习率

# ── 3. mAP 恒为 0：三大元凶（各 3 分钟）────────────────────────────────
#   A. 预测类别整体 ±1 重算 mAP            -> 突然正常 = 类别 ID 偏移
#      指纹：混淆矩阵质量在偏移 k 格的次对角线上（detect_label_shift）
#   B. 框按 cxcywh/xywh/yxyx -> xyxy 各试一遍 -> IoU 从 0.0x 跳到 0.8+ = 格式错
#      指纹：**IoU 恰好为 0 的比例 ~100%**（定位不准时该比例 ~0%）
#            yxyx 的指纹是 corr(pred_cx, gt_cy) ~ 1 而 corr(pred_cx, gt_cx) ~ 0
#   C. diff 两侧类别表的**顺序**（不只是集合）-> 混淆矩阵是置换矩阵 = 类别表不一致
#   三条都过了才允许怀疑模型。「mAP 恰好是 0 而不是 0.3」本身就是线索。

# ── 4. 指标掉了但没归零（最危险，因为像「没调好」）──────────────────────
#   通道顺序 BGR/RGB : **颜色语义类崩、中性色类不掉**；x = x[..., ::-1] 后恢复
#   归一化 mean/std  : **所有类均匀掉点**；dump 一个 batch 比对通道均值
#   resize 插值/对齐 : **小目标掉得多、大目标几乎不掉**；同图两边逐元素 diff
#   ⚠️ 组合 bug 会污染指纹 -> 一次只改一个变量重测（单变量原则）

# ── 5. 训练好、验证崩 ──────────────────────────────────────────────────
#   决定性实验：**把训练集喂进验证的那条代码路径**
#     同样崩 -> 管线问题（预处理不一致 / 忘了 eval() / 类别表）
#     依然好 -> 真泛化问题（过拟合 / 捷径特征）
#   捷径特征的判据：按采集批次分桶，桶内好、跨桶崩

# ── 6. 分布式 ──────────────────────────────────────────────────────────
#   BN 不同步        : per-GPU batch < 16 时换 SyncBatchNorm
#   seed 全卡相同     : seed = base + rank；DataLoader 再叠 worker_id
#   忘了 set_epoch    : 每个 epoch 开头 sampler.set_epoch(epoch)  <- 最常忘
#   数据分片重叠      : len(sampler)*world_size 与数据集大小对比；评测结果按 id 去重
#   梯度累积          : loss 要除以累积步数，否则等效 lr 变成 k 倍

# ── 7. 复现失败的排查顺序（每步都比下一步便宜）─────────────────────────
#   ① 种子方差（检测任务 ±0.2-0.5 mAP 是常态）-> 先跑 3-5 个种子看分布
#   ② 代码版本（git status / pip freeze diff）
#   ③ 数据版本（标注文件 md5；数据是持续回流的）
#   ④ 随机性来源（python/numpy/框架/cudnn/worker；cudnn.benchmark 依赖硬件状态）
#   ⑤ 环境（驱动/CUDA/框架/卡型号 —— 换卡会改变 kernel 与浮点累加顺序）
#   ⑥ 最后才是「代码里有 bug」
#   发布候选版本必须在确定性模式下重跑并归档（固定 seed + deterministic + 镜像 + 数据快照）
'''
print(RECIPE)
for tok in ['单 batch 过拟合', 'log-sum-exp', 'float32 z≈17', 'IoU 恰好为 0',
            '颜色语义类崩', 'set_epoch', '种子方差', 'n_pos == 0']:
    assert tok in RECIPE, tok
print('✅ 手册覆盖：不变量测试 / 单batch过拟合 / NaN / mAP=0 三元凶 / 指纹 / 验证崩 / 分布式 / 复现')

### 小结

- **调试的进展不是「试了很多东西」，而是「排除了很多可能」。**
  每一段链路都要有一个「必须成立的等式」：`decode∘encode = id`、`L(GT,GT)≈0`、
  数值梯度 == 解析梯度、`GT 当预测 -> mAP = 1.0`。**把它们写成常驻单元测试。**
- **「先把一个 batch 过拟合」是唯一无条件必须成立的实验。**
  它成立 ⇒ 数据/编码/前向/损失/反传/优化器整块可用，搜索空间立刻砍半；
  它不成立 ⇒ bug 一定在前六段。做之前必须关掉增强/dropout/weight decay/**warmup 与衰减**/EMA。
- **检测特有：必须同时打印正样本数。** RetinaNet 默认 anchor（P3–P7）下，
  **≤ 20 px 的交通标志拿不到任何 IoU ≥ 0.5 的正样本**——它对分类损失完全不可见。
  加一个 P2 层（stride 4 / base 16）能把 12–20 px 救回来。
- **NaN 的第一个问题永远是「第几个 iteration 出现的」。**
  0–1 步 ⇒ 实现或数据；几百步后 ⇒ 发散；固定 seed 后总在同一步 ⇒ 某个样本。
  手写损失一律用 log-sum-exp（朴素式断点：**float32 z≈17，float64 z≈37**），
  IoU 分母一律加 `eps`（真实标注里 `x1==x2` 的退化框并不罕见）。
- **mAP 恒为 0 的三大元凶**：类别 ID 偏移 / 坐标格式弄反 / 类别表不一致，
  各 3 分钟可查完。「mAP 恰好是 0 而不是 0.3」本身就是线索——
  **模型能力问题不会让指标精确归零。**
- **指纹思维把调试从「顺序搜索」变成「哈希查表」。** 构造指纹的通用方法是**找不对称**：
  通道顺序 → 颜色维度不对称（颜色类崩、中性色类不掉）；
  resize 不一致 → 尺寸维度不对称；类别表错位 → 类别维度呈置换结构；
  坐标格式 → **IoU 恰好为 0 的比例 ≈ 100%**（定位不准时 ≈ 0%）。
- **「验证集上差」不等于「过拟合」。** 决定性实验是把训练集喂进验证的代码路径：
  同样崩 = 管线问题，依然好 = 真泛化问题。这个实验 20 分钟能做完，结论是二值的。
- **复现失败先怀疑方差，不要先怀疑 bug。** 检测任务同配置不同种子的 mAP 波动
  典型是 ±0.2–0.5 个点，这一条排除掉的「bug」占了大半。

下一站：**模块 04 · 项目叙事** —— 把这些排查过程讲成让人信服的故事。